# 24H Journal Revision — Clean Transformer Benchmark

This clean notebook runs the **24h forecasting branch** using the former 24h code as the base, but fixes the revision issues:

1. Uses the same 24h dataset and features: **16 SHARP magnetic features + GOES-derived flare-history features**.
2. Builds chronological lookback sequences.
3. Trains **LSTM, BiLSTM, Transformer, and DLSTM**.
4. Uses scaled inputs consistently.
5. Tunes decision thresholds on the **validation set only**.
6. Selects weighted ensemble weights using **validation TSS only**.
7. Saves a manuscript-ready results table for the 24h section.

No stale/manual table. No test-set weight selection.

In [1]:
# ============================================================
# 0. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed fixed:", SEED)

TensorFlow: 2.20.0
Seed fixed: 42


In [3]:
# ============================================================
# 2. Experiment configuration
# ============================================================

DATA_PATH_24H = "/content/drive/MyDrive/AR_Stratified/24H_FINAL_CLEAN_READY.csv"
OUT_DIR = "/content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER"
os.makedirs(OUT_DIR, exist_ok=True)

TIME_COL = "T_REC_dt"
AR_COL = "NOAA_AR"
LABEL_COL = "label_MX_1d"

LOOKBACK_24H = 4          # 4 sampled states; physical span is measured below from the actual cadence.
MA_WINDOW = 3
EPOCHS = 30
BATCH_SIZE = 128

MAG_FEATURES = [
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH", "MEANJZD",
    "TOTUSJZ", "MEANALP", "MEANJZH", "ABSNJZH", "SAVNCPP",
    "MEANSHR", "SHRGT45", "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH"
]

FLARE_HISTORY_FEATURES = [
    "flare_count_past_24h",
    "time_since_last_flare_hours_24h",
    "max_peak_flux_past_24h",
    "mean_peak_flux_past_24h",
    "flare_activity_index_past_24h"
]

ALL_FEATURES = MAG_FEATURES + FLARE_HISTORY_FEATURES

print("DATA_PATH_24H:", DATA_PATH_24H)
print("OUT_DIR:", OUT_DIR)
print("LOOKBACK_24H:", LOOKBACK_24H)
print("MA_WINDOW:", MA_WINDOW)
print("Magnetic features:", len(MAG_FEATURES))
print("Flare-history features:", len(FLARE_HISTORY_FEATURES))
print("Total features:", len(ALL_FEATURES))

DATA_PATH_24H: /content/drive/MyDrive/AR_Stratified/24H_FINAL_CLEAN_READY.csv
OUT_DIR: /content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER
LOOKBACK_24H: 4
MA_WINDOW: 3
Magnetic features: 16
Flare-history features: 5
Total features: 21


In [4]:
# ============================================================
# 3. Load, clean, and inspect dataset
# ============================================================

df = pd.read_csv(DATA_PATH_24H)

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [TIME_COL, AR_COL, LABEL_COL] + ALL_FEATURES
missing_cols = [c for c in required_cols if c not in df.columns]
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

for col in ALL_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required_cols).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df.sort_values([AR_COL, TIME_COL]).reset_index(drop=True)

print("Clean shape:", df.shape)
print("Label distribution:", df[LABEL_COL].value_counts().to_dict())
print("Positive prevalence:", df[LABEL_COL].mean())

display(df[[TIME_COL, AR_COL, LABEL_COL] + ALL_FEATURES[:8]].head())

Raw shape: (22355, 26)
Columns: ['T_REC_dt', 'year', 'NOAA_AR', 'MEANGBZ', 'MEANGAM', 'MEANGBT', 'MEANGBH', 'MEANJZD', 'TOTUSJZ', 'MEANALP', 'MEANJZH', 'ABSNJZH', 'SAVNCPP', 'MEANSHR', 'SHRGT45', 'R_VALUE', 'USFLUX', 'TOTPOT', 'TOTUSJH', 'label_MX_1d', 'flare_count_past_24h', 'time_since_last_flare_hours_24h', 'max_peak_flux_past_24h', 'mean_peak_flux_past_24h', 'flare_activity_index_past_24h', 'had_flare_past_24h']
Clean shape: (5493, 26)
Label distribution: {0: 4894, 1: 599}
Positive prevalence: 0.10904787911887857


,T_REC_dt,NOAA_AR,label_MX_1d,MEANGBZ,MEANGAM,MEANGBT,MEANGBH,MEANJZD,TOTUSJZ,MEANALP,MEANJZH
0,2010-05-04 12:00:00,11063,0,114.931,23.572,111.969,48.270,-0.446712,1.144288e+12,-0.006346,-0.003990
1,2010-05-04 12:00:00,11066,0,163.612,33.349,158.310,83.826,2.161599,9.491178e+11,-0.047176,-0.023881
2,2010-05-06 12:00:00,11067,0,120.344,30.754,123.132,52.503,0.809853,5.854035e+12,-0.000631,-0.000130
3,2010-05-04 12:00:00,11068,0,68.033,20.239,66.450,27.658,-0.044283,2.832238e+12,0.004054,0.001657
4,2010-05-06 12:00:00,11068,0,109.701,29.563,109.877,44.013,0.177906,7.031297e+12,0.024635,0.006839


In [5]:
# ============================================================
# 4. Cadence and lookback diagnostics
# ============================================================

cadence_hours = (
    df.sort_values([AR_COL, TIME_COL])
      .groupby(AR_COL)[TIME_COL]
      .diff()
      .dt.total_seconds() / 3600.0
).dropna()

print("Cadence diagnostics in hours:")
display(cadence_hours.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

median_cadence = float(cadence_hours.median()) if len(cadence_hours) else np.nan
approx_span = (LOOKBACK_24H - 1) * median_cadence if not np.isnan(median_cadence) else np.nan

print(f"Median cadence: {median_cadence:.2f} hours")
print(f"Approximate span between first and final input timestamp: ({LOOKBACK_24H}-1) × {median_cadence:.2f}h = {approx_span:.2f}h")
print("Note: the 24h label horizon refers to the future prediction window, not necessarily the exact input span.")
print(f"Rows with gaps > 2 × median cadence: {int((cadence_hours > 2 * median_cadence).sum()) if not np.isnan(median_cadence) else 'NA'}")

Cadence diagnostics in hours:


,T_REC_dt
count,4137.000000
mean,36.142132
std,28.435667
min,24.000000
25%,24.000000
50%,24.000000
75%,24.000000
90%,72.000000
95%,96.000000
99%,168.000000


Median cadence: 24.00 hours
Approximate span between first and final input timestamp: (4-1) × 24.00h = 72.00h
Note: the 24h label horizon refers to the future prediction window, not necessarily the exact input span.
Rows with gaps > 2 × median cadence: 480


In [6]:
# ============================================================
# 5. Build chronological sequences
# ============================================================

def build_sequences(frame, feature_cols, lookback):
    X_seq, y_seq, t_seq, ar_seq = [], [], [], []

    for ar, group in frame.groupby(AR_COL):
        group = group.sort_values(TIME_COL).reset_index(drop=True)

        if len(group) < lookback:
            continue

        values = group[feature_cols].values.astype(np.float32)
        labels = group[LABEL_COL].values.astype(int)
        times = group[TIME_COL].values

        for i in range(lookback - 1, len(group)):
            X_seq.append(values[i - lookback + 1:i + 1])
            y_seq.append(labels[i])
            t_seq.append(times[i])
            ar_seq.append(ar)

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(y_seq, dtype=int),
        np.array(t_seq),
        np.array(ar_seq)
    )


X_seq, y_seq, t_seq, ar_seq = build_sequences(df, ALL_FEATURES, LOOKBACK_24H)

print("X_seq:", X_seq.shape)
print("y_seq:", y_seq.shape)
print("Positive count:", int(y_seq.sum()))
print("Positive prevalence:", y_seq.mean())

X_seq: (2342, 4, 21)
y_seq: (2342,)
Positive count: 344
Positive prevalence: 0.14688300597779674


In [7]:
# ============================================================
# 6. Chronological split and scaling
# ============================================================

order = np.argsort(t_seq)

X_seq = X_seq[order]
y_seq = y_seq[order]
t_seq = t_seq[order]
ar_seq = ar_seq[order]

n_total = len(X_seq)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

X_train_seq = X_seq[:n_train]
y_train_seq = y_seq[:n_train]

X_val_seq = X_seq[n_train:n_train + n_val]
y_val_seq = y_seq[n_train:n_train + n_val]

X_test_seq = X_seq[n_train + n_val:]
y_test_seq = y_seq[n_train + n_val:]

t_train = t_seq[:n_train]
t_val = t_seq[n_train:n_train + n_val]
t_test = t_seq[n_train + n_val:]

ar_train = set(ar_seq[:n_train])
ar_val = set(ar_seq[n_train:n_train + n_val])
ar_test = set(ar_seq[n_train + n_val:])

print("Train:", X_train_seq.shape, "Pos:", int(y_train_seq.sum()))
print("Val  :", X_val_seq.shape, "Pos:", int(y_val_seq.sum()))
print("Test :", X_test_seq.shape, "Pos:", int(y_test_seq.sum()))

print("\nTemporal range:")
print("Train:", pd.to_datetime(t_train.min()), "→", pd.to_datetime(t_train.max()))
print("Val  :", pd.to_datetime(t_val.min()), "→", pd.to_datetime(t_val.max()))
print("Test :", pd.to_datetime(t_test.min()), "→", pd.to_datetime(t_test.max()))

print("\nActive-region overlap across chronological splits:")
print("Train ∩ Val :", len(ar_train & ar_val))
print("Val ∩ Test  :", len(ar_val & ar_test))
print("Train ∩ Test:", len(ar_train & ar_test))

scaler = StandardScaler()
n_features = X_train_seq.shape[-1]

X_train_scaled = scaler.fit_transform(
    X_train_seq.reshape(-1, n_features)
).reshape(X_train_seq.shape)

X_val_scaled = scaler.transform(
    X_val_seq.reshape(-1, n_features)
).reshape(X_val_seq.shape)

X_test_scaled = scaler.transform(
    X_test_seq.reshape(-1, n_features)
).reshape(X_test_seq.shape)

joblib.dump(scaler, os.path.join(OUT_DIR, "scaler_24h_clean_transformer.joblib"))

print("\nScaled shapes:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled  :", X_val_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

Train: (1639, 4, 21) Pos: 174
Val  : (351, 4, 21) Pos: 63
Test : (352, 4, 21) Pos: 107

Temporal range:
Train: 2010-05-08 12:00:00 → 2023-02-22 12:00:00
Val  : 2023-02-24 12:00:00 → 2024-02-16 12:00:00
Test : 2024-02-16 12:00:00 → 2025-04-18 12:00:00

Active-region overlap across chronological splits:
Train ∩ Val : 1
Val ∩ Test  : 2
Train ∩ Test: 0

Scaled shapes:
X_train_scaled: (1639, 4, 21)
X_val_scaled  : (351, 4, 21)
X_test_scaled : (352, 4, 21)


In [8]:
# ============================================================
# 7. Metric helpers
# ============================================================

def tss_hss_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    tss = tpr - fpr

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn) + 1e-9)
    hss = numerator / denominator

    return float(tss), float(hss)


def tune_threshold_by_tss(y_true, prob):
    thresholds = np.linspace(0, 1, 1001)
    best_thr = 0.5
    best_tss = -999.0

    for thr in thresholds:
        y_pred = (prob >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)

        if tss > best_tss:
            best_tss = tss
            best_thr = thr

    return float(best_thr), float(best_tss)


def evaluate_probs_clean(model_name, y_val, val_prob, y_test, test_prob):
    threshold, val_tss = tune_threshold_by_tss(y_val, val_prob)

    y_pred = (test_prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tss, hss = tss_hss_from_cm(cm)
    tn, fp, fn, tp = cm.ravel()

    return {
        "Model": model_name,
        "Threshold": float(threshold),
        "Val_TSS": float(val_tss),
        "Test_TSS": float(tss),
        "HSS": float(hss),
        "ROC_AUC": float(roc_auc_score(y_test, test_prob)),
        "PR_AUC": float(average_precision_score(y_test, test_prob)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }, y_pred


def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


class_weight_dict = get_class_weights(y_train_seq)
print("Class weights:", class_weight_dict)

Class weights: {0: 0.5593856655290103, 1: 4.709770114942529}


In [9]:
# ============================================================
# 8. Model builders
# ============================================================

def make_callbacks(patience=5):
    return [
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-5,
            verbose=1
        )
    ]


def build_lstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.0),
        layers.Dropout(0.3),
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def build_bilstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.0)),
        layers.Dropout(0.3),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0)),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def transformer_encoder_block(x, num_heads=4, key_dim=32, ff_dim=128, dropout=0.20):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x


def build_transformer_model(input_shape, num_blocks=2, num_heads=4, key_dim=32, ff_dim=128, dropout=0.20, dense_units=64):
    inp = layers.Input(shape=input_shape, name="sequence_input")

    x = layers.Dense(64, activation="relu", name="feature_projection")(inp)

    for _ in range(num_blocks):
        x = transformer_encoder_block(
            x,
            num_heads=num_heads,
            key_dim=key_dim,
            ff_dim=ff_dim,
            dropout=dropout
        )

    x = layers.GlobalAveragePooling1D(name="temporal_pooling")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)

    out = layers.Dense(1, activation="sigmoid", name="flare_probability")(x)

    model = models.Model(inp, out, name="Transformer_24h")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def moving_average_decomposition(X, window=3):
    trend = np.zeros_like(X)
    kernel = np.ones(window, dtype=np.float32) / float(window)

    for feature_idx in range(X.shape[2]):
        for sample_idx in range(X.shape[0]):
            trend[sample_idx, :, feature_idx] = np.convolve(
                X[sample_idx, :, feature_idx],
                kernel,
                mode="same"
            )

    residual = X - trend
    return trend, residual


def build_dlstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.0),
        layers.Dropout(0.3),
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model

In [10]:
# ============================================================
# 9. Train LSTM
# ============================================================

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])

lstm_model = build_lstm_model(input_shape)
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train_scaled,
    y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_lstm = lstm_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_lstm = lstm_model.predict(X_test_scaled, verbose=0).ravel()

results_lstm, y_pred_lstm = evaluate_probs_clean(
    "LSTM",
    y_val_seq,
    val_prob_lstm,
    y_test_seq,
    test_prob_lstm
)

display(pd.DataFrame([results_lstm]))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 128)         │        76,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,321 (501.25 KB)

 Trainable params: 128,321 (501.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - auc: 0.7274 - loss: 0.6561 - val_auc: 0.7440 - val_loss: 0.7126 - learning_rate: 0.0010
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.7713 - loss: 0.5892 - val_auc: 0.7451 - val_loss: 0.7401 - learning_rate: 0.0010
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.7955 - loss: 0.5470 - val_auc: 0.7494 - val_loss: 0.7111 - learning_rate: 0.0010
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.8042 - loss: 0.5325 - val_auc: 0.7595 - val_loss: 0.6891 - learning_rate: 0.0010
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.8214 - loss: 0.5143 - val_auc: 0.7773 - val_loss: 0.6937 - learning_rate: 0.0010
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.8293 - loss: 0.5016 - val_auc: 0.7939 - val_loss: 0.6798 - learning_rate: 0.0010
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - auc: 0.8421 - loss: 0.4910 - val_auc: 0.8096 - val_loss: 0.6755 - learning_rate: 0.0010
Epoch 8/30
13

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,LSTM,0.59,0.560516,0.427046,0.362216,0.801106,0.66806,0.794393,0.485714,0.602837,155,90,22,85


In [11]:
# ============================================================
# 10. Train BiLSTM
# ============================================================

bilstm_model = build_bilstm_model(input_shape)
bilstm_model.summary()

history_bilstm = bilstm_model.fit(
    X_train_scaled,
    y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_bilstm = bilstm_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_bilstm = bilstm_model.predict(X_test_scaled, verbose=0).ravel()

results_bilstm, y_pred_bilstm = evaluate_probs_clean(
    "BiLSTM",
    y_val_seq,
    val_prob_bilstm,
    y_test_seq,
    test_prob_bilstm
)

display(pd.DataFrame([results_bilstm]))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 4, 256)         │       153,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 4, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,113 (1.23 MB)

 Trainable params: 322,113 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 129ms/step - auc: 0.7399 - loss: 0.6247 - val_auc: 0.7808 - val_loss: 0.7563 - learning_rate: 0.0010
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - auc: 0.8238 - loss: 0.5161 - val_auc: 0.7948 - val_loss: 0.6903 - learning_rate: 0.0010
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - auc: 0.8425 - loss: 0.4907 - val_auc: 0.8088 - val_loss: 0.6778 - learning_rate: 0.0010
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - auc: 0.8484 - loss: 0.4786 - val_auc: 0.8208 - val_loss: 0.6891 - learning_rate: 0.0010
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - auc: 0.8655 - loss: 0.4585 - val_auc: 0.8304 - val_loss: 0.6795 - learning_rate: 0.0010
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - auc: 0.8664 - loss: 0.4585 - val_auc: 0.8350 - val_loss: 0.6728 - learning_rate: 0.0010
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - auc: 0.8829 - loss: 0.4260 - val_auc: 0.8392 - val_loss: 0.6479 - learning_rate: 0.0010
Epoch 8/30
1

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,BiLSTM,0.565,0.568948,0.43704,0.359458,0.80637,0.687897,0.841121,0.47619,0.608108,146,99,17,90


In [12]:
# ============================================================
# 11. Train Transformer
# ============================================================

transformer_model = build_transformer_model(input_shape)
transformer_model.summary()

history_transformer = transformer_model.fit(
    X_train_scaled,
    y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(6),
    shuffle=False,
    verbose=1
)

val_prob_transformer = transformer_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_transformer = transformer_model.predict(X_test_scaled, verbose=0).ravel()

results_transformer, y_pred_transformer = evaluate_probs_clean(
    "Transformer",
    y_val_seq,
    val_prob_transformer,
    y_test_seq,
    test_prob_transformer
)

display(pd.DataFrame([results_transformer]))

Model: "Transformer_24h"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 4, 21)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_projection  │ (None, 4, 64)     │      1,408 │ sequence_input[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 4, 64)     │     33,216 │ feature_projecti… │
│ (MultiHeadAttentio… │                   │            │ feature_projecti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 4, 64)     │          0 │ feature_projecti… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 4, 64)     │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 4, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 4, 128)    │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 4, 64)     │      8,256 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 4, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 64)     │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 4, 64)     │     33,216 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 4, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 64)     │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 4, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 4, 128)    │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 4, 64)     │      8,256 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 4, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 64)     │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_pooling    │ (None, 64)        │          0 │ layer_normalizat

 Total params: 105,729 (413.00 KB)

 Trainable params: 105,729 (413.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 240ms/step - auc: 0.7040 - loss: 0.6463 - val_auc: 0.7553 - val_loss: 0.7754 - learning_rate: 0.0010
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - auc: 0.7886 - loss: 0.5447 - val_auc: 0.7558 - val_loss: 0.8928 - learning_rate: 0.0010
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - auc: 0.8097 - loss: 0.5182 - val_auc: 0.7625 - val_loss: 0.7832 - learning_rate: 0.0010
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - auc: 0.8333 - loss: 0.4902 - val_auc: 0.7840 - val_loss: 0.7363 - learning_rate: 0.0010
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - auc: 0.8402 - loss: 0.4841 - val_auc: 0.7764 - val_loss: 0.7305 - learning_rate: 0.0010
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - auc: 0.8490 - loss: 0.4664 - val_auc: 0.7892 - val_loss: 0.7473 - learning_rate: 0.0010
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 205ms/step - auc: 0.8529 - loss: 0.4644 - val_auc: 0.8007 - val_loss: 0.7176 - learning_rate: 0.0010
Epoch

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,Transformer,0.669,0.487599,0.357658,0.292269,0.759184,0.596968,0.794393,0.442708,0.568562,138,107,22,85


In [13]:
# ============================================================
# 12. DLSTM decomposition and training
# ============================================================

trend_train, res_train = moving_average_decomposition(X_train_scaled, window=MA_WINDOW)
trend_val, res_val = moving_average_decomposition(X_val_scaled, window=MA_WINDOW)
trend_test, res_test = moving_average_decomposition(X_test_scaled, window=MA_WINDOW)

X_train_dlstm = np.concatenate([trend_train, res_train], axis=2)
X_val_dlstm = np.concatenate([trend_val, res_val], axis=2)
X_test_dlstm = np.concatenate([trend_test, res_test], axis=2)

print("DLSTM input shape:", X_train_dlstm.shape)

dlstm_model = build_dlstm_model((X_train_dlstm.shape[1], X_train_dlstm.shape[2]))
dlstm_model.summary()

history_dlstm = dlstm_model.fit(
    X_train_dlstm,
    y_train_seq,
    validation_data=(X_val_dlstm, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_dlstm = dlstm_model.predict(X_val_dlstm, verbose=0).ravel()
test_prob_dlstm = dlstm_model.predict(X_test_dlstm, verbose=0).ravel()

results_dlstm, y_pred_dlstm = evaluate_probs_clean(
    "DLSTM",
    y_val_seq,
    val_prob_dlstm,
    y_test_seq,
    test_prob_dlstm
)

display(pd.DataFrame([results_dlstm]))

DLSTM input shape: (1639, 4, 42)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 4, 128)         │        87,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 4, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 139,073 (543.25 KB)

 Trainable params: 139,073 (543.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - auc: 0.6887 - loss: 0.6639 - val_auc: 0.7588 - val_loss: 0.6775 - learning_rate: 0.0010
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - auc: 0.7716 - loss: 0.5867 - val_auc: 0.7535 - val_loss: 0.7127 - learning_rate: 0.0010
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - auc: 0.7970 - loss: 0.5452
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.7943 - loss: 0.5417 - val_auc: 0.7561 - val_loss: 0.7051 - learning_rate: 0.0010
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - auc: 0.7991 - loss: 0.5389 - val_auc: 0.7586 - val_loss: 0.6942 - learning_rate: 5.0000e-04
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - auc: 0.8083 - loss: 0.5241 - val_auc: 0.7621 - val_loss: 0.6959 - learning_rate: 5.0000e-04
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - auc: 0.8145 - loss: 0.5155 - val_auc: 0.7657 - val_loss: 0.6971 - learning_rate: 5.0000e-

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,DLSTM,0.566,0.551091,0.396224,0.318982,0.803166,0.677047,0.841121,0.452261,0.588235,136,109,17,90


In [14]:
# ============================================================
# 13. Final clean 24h table: individual + ensembles
# ============================================================

final_rows = [
    results_lstm,
    results_bilstm,
    results_transformer,
    results_dlstm
]

# ------------------------------------------------------------
# Simple average ensemble: LSTM + BiLSTM + DLSTM
# ------------------------------------------------------------

val_prob_simple = (val_prob_lstm + val_prob_bilstm + val_prob_dlstm) / 3.0
test_prob_simple = (test_prob_lstm + test_prob_bilstm + test_prob_dlstm) / 3.0

results_simple, y_pred_simple = evaluate_probs_clean(
    "Simple/Avg Ensemble",
    y_val_seq,
    val_prob_simple,
    y_test_seq,
    test_prob_simple
)

final_rows.append(results_simple)


# ------------------------------------------------------------
# Weighted ensemble search
# Selected by VALIDATION TSS only.
# ------------------------------------------------------------

weight_sets = [
    (0.20, 0.40, 0.40),
    (0.20, 0.30, 0.50),
    (0.15, 0.35, 0.50),
    (0.25, 0.35, 0.40),
    (0.30, 0.30, 0.40),
    (0.10, 0.45, 0.45),
    (0.33, 0.33, 0.34),
    (0.25, 0.25, 0.50),
    (0.40, 0.30, 0.30),
    (1/3, 1/3, 1/3)
]

weighted_rows = []
weighted_preds = {}

for w_lstm, w_bilstm, w_dlstm in weight_sets:
    val_prob_weighted = (
        w_lstm * val_prob_lstm +
        w_bilstm * val_prob_bilstm +
        w_dlstm * val_prob_dlstm
    )

    test_prob_weighted = (
        w_lstm * test_prob_lstm +
        w_bilstm * test_prob_bilstm +
        w_dlstm * test_prob_dlstm
    )

    model_name = f"Weighted Ensemble ({w_lstm:.2f},{w_bilstm:.2f},{w_dlstm:.2f})"

    result, y_pred_weighted = evaluate_probs_clean(
        model_name,
        y_val_seq,
        val_prob_weighted,
        y_test_seq,
        test_prob_weighted
    )

    weighted_rows.append(result)
    weighted_preds[model_name] = y_pred_weighted

weighted_df = pd.DataFrame(weighted_rows).sort_values(
    ["Val_TSS", "Test_TSS"],
    ascending=[False, False]
).reset_index(drop=True)

best_weighted_result = weighted_df.iloc[0].to_dict()
best_weighted_name = best_weighted_result["Model"]
y_pred_best_weighted = weighted_preds[best_weighted_name]

final_rows.append(best_weighted_result)


# ------------------------------------------------------------
# Stacking ensemble: logistic regression on validation predictions
# ------------------------------------------------------------

X_meta_val = np.column_stack([
    val_prob_lstm,
    val_prob_bilstm,
    val_prob_dlstm
])

X_meta_test = np.column_stack([
    test_prob_lstm,
    test_prob_bilstm,
    test_prob_dlstm
])

stack_model = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    random_state=SEED
)

stack_model.fit(X_meta_val, y_val_seq)

val_prob_stack = stack_model.predict_proba(X_meta_val)[:, 1]
test_prob_stack = stack_model.predict_proba(X_meta_test)[:, 1]

results_stack, y_pred_stack = evaluate_probs_clean(
    "Stacking Ensemble",
    y_val_seq,
    val_prob_stack,
    y_test_seq,
    test_prob_stack
)

final_rows.append(results_stack)


# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------

final_24h_df = pd.DataFrame(final_rows).sort_values(
    "Test_TSS",
    ascending=False
).reset_index(drop=True)

display(final_24h_df.round(6))

print("\nWeighted ensemble search, selected by validation TSS:")
display(weighted_df.round(6))

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,BiLSTM,0.565,0.568948,0.437040,0.359458,0.806370,0.687897,0.841121,0.476190,0.608108,146,99,17,90
1,LSTM,0.590,0.560516,0.427046,0.362216,0.801106,0.668060,0.794393,0.485714,0.602837,155,90,22,85
2,"Weighted Ensemble (0.20,0.30,0.50)",0.593,0.580357,0.418882,0.353712,0.807858,0.685879,0.794393,0.480226,0.598592,153,92,22,85
3,Stacking Ensemble,0.473,0.582341,0.417166,0.346866,0.809384,0.685773,0.813084,0.472826,0.597938,148,97,20,87
4,Simple/Avg Ensemble,0.585,0.569940,0.406637,0.341098,0.807934,0.685540,0.794393,0.472222,0.592334,150,95,22,85
5,DLSTM,0.566,0.551091,0.396224,0.318982,0.803166,0.677047,0.841121,0.452261,0.588235,136,109,17,90
6,Transformer,0.669,0.487599,0.357658,0.292269,0.759184,0.596968,0.794393,0.442708,0.568562,138,107,22,85



Weighted ensemble search, selected by validation TSS:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,"Weighted Ensemble (0.20,0.30,0.50)",0.593,0.580357,0.418882,0.353712,0.807858,0.685879,0.794393,0.480226,0.598592,153,92,22,85
1,"Weighted Ensemble (0.15,0.35,0.50)",0.592,0.576885,0.422964,0.357955,0.808049,0.686854,0.794393,0.482955,0.600707,154,91,22,85
2,"Weighted Ensemble (0.25,0.25,0.50)",0.588,0.576885,0.410719,0.345284,0.807591,0.684537,0.794393,0.474860,0.594406,151,94,22,85
3,"Weighted Ensemble (0.25,0.35,0.40)",0.589,0.573413,0.418882,0.353712,0.808201,0.686757,0.794393,0.480226,0.598592,153,92,22,85
4,"Weighted Ensemble (0.10,0.45,0.45)",0.598,0.573413,0.417700,0.355081,0.808697,0.687265,0.785047,0.482759,0.597865,155,90,23,84
5,"Weighted Ensemble (0.33,0.33,0.34)",0.587,0.573413,0.410719,0.345284,0.807858,0.685569,0.794393,0.474860,0.594406,151,94,22,85
6,"Weighted Ensemble (0.30,0.30,0.40)",0.586,0.573413,0.402556,0.336930,0.807820,0.685528,0.794393,0.469613,0.590278,149,96,22,85
7,"Weighted Ensemble (0.20,0.40,0.40)",0.589,0.569940,0.414801,0.349489,0.808774,0.686940,0.794393,0.477528,0.596491,152,93,22,85
8,"Weighted Ensemble (0.33,0.33,0.33)",0.585,0.569940,0.406637,0.341098,0.807934,0.685540,0.794393,0.472222,0.592334,150,95,22,85
9,"Weighted Ensemble (0.40,0.30,0.30)",0.569,0.568452,0.391493,0.323397,0.807553,0.684061,0.803738,0.459893,0.585034,144,101,21,86


In [15]:
# ============================================================
# 14. Save clean outputs
# ============================================================

final_24h_df.to_csv(
    os.path.join(OUT_DIR, "FINAL_CLEAN_24H_RESULTS_WITH_TRANSFORMER.csv"),
    index=False
)

weighted_df.to_csv(
    os.path.join(OUT_DIR, "FINAL_24H_WEIGHTED_ENSEMBLE_SEARCH_BY_VAL_TSS.csv"),
    index=False
)

np.save(os.path.join(OUT_DIR, "y_val_24h.npy"), y_val_seq)
np.save(os.path.join(OUT_DIR, "y_test_24h.npy"), y_test_seq)

arrays_to_save = {
    "val_prob_lstm": val_prob_lstm,
    "test_prob_lstm": test_prob_lstm,
    "y_pred_lstm": y_pred_lstm,
    "val_prob_bilstm": val_prob_bilstm,
    "test_prob_bilstm": test_prob_bilstm,
    "y_pred_bilstm": y_pred_bilstm,
    "val_prob_transformer": val_prob_transformer,
    "test_prob_transformer": test_prob_transformer,
    "y_pred_transformer": y_pred_transformer,
    "val_prob_dlstm": val_prob_dlstm,
    "test_prob_dlstm": test_prob_dlstm,
    "y_pred_dlstm": y_pred_dlstm,
    "val_prob_simple": val_prob_simple,
    "test_prob_simple": test_prob_simple,
    "y_pred_simple": y_pred_simple,
    "y_pred_best_weighted": y_pred_best_weighted,
    "val_prob_stack": val_prob_stack,
    "test_prob_stack": test_prob_stack,
    "y_pred_stack": y_pred_stack,
}

for name, arr in arrays_to_save.items():
    np.save(os.path.join(OUT_DIR, f"{name}.npy"), arr)

joblib.dump(stack_model, os.path.join(OUT_DIR, "stacking_model_24h.joblib"))

print("Saved all clean outputs to:")
print(OUT_DIR)

Saved all clean outputs to:
/content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER


In [16]:
# ============================================================
# 15. Bootstrap confidence interval for best TSS model
# ============================================================

def bootstrap_tss_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)

    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred).astype(int)

    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_pred[idx]

        if len(np.unique(yt)) < 2:
            continue

        cm = confusion_matrix(yt, yp, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)
        scores.append(tss)

    scores = np.array(scores)

    return {
        "mean": float(scores.mean()),
        "ci_low": float(np.percentile(scores, 2.5)),
        "ci_high": float(np.percentile(scores, 97.5)),
        "n_boot_valid": int(len(scores))
    }, scores


best_model_name = final_24h_df.iloc[0]["Model"]

pred_lookup = {
    "LSTM": y_pred_lstm,
    "BiLSTM": y_pred_bilstm,
    "Transformer": y_pred_transformer,
    "DLSTM": y_pred_dlstm,
    "Simple/Avg Ensemble": y_pred_simple,
    best_weighted_name: y_pred_best_weighted,
    "Stacking Ensemble": y_pred_stack
}

best_pred = pred_lookup[best_model_name]

best_ci, best_boot_scores = bootstrap_tss_ci(
    y_test_seq,
    best_pred,
    n_boot=2000,
    seed=SEED
)

print("Best model:", best_model_name)
print("Bootstrap TSS CI:", best_ci)

pd.DataFrame([{
    "Model": best_model_name,
    "Bootstrap_Mean_TSS": best_ci["mean"],
    "CI_Low": best_ci["ci_low"],
    "CI_High": best_ci["ci_high"],
    "N_Boot_Valid": best_ci["n_boot_valid"]
}]).to_csv(
    os.path.join(OUT_DIR, "BEST_24H_MODEL_BOOTSTRAP_TSS_CI.csv"),
    index=False
)

np.save(
    os.path.join(OUT_DIR, "best_24h_model_bootstrap_tss_scores.npy"),
    best_boot_scores
)

Best model: BiLSTM
Bootstrap TSS CI: {'mean': 0.43748164338741696, 'ci_low': 0.3446322761801376, 'ci_high': 0.5241362594679626, 'n_boot_valid': 2000}


In [17]:
# ============================================================
# 16. McNemar comparisons: best model vs baselines
# ============================================================

from math import erf, sqrt

def mcnemar_test(y_true, pred_a, pred_b):
    y_true = np.array(y_true).astype(int)
    pred_a = np.array(pred_a).astype(int)
    pred_b = np.array(pred_b).astype(int)

    a_correct = pred_a == y_true
    b_correct = pred_b == y_true

    b = int(np.sum((a_correct == 1) & (b_correct == 0)))
    c = int(np.sum((a_correct == 0) & (b_correct == 1)))

    n = b + c

    if n == 0:
        chi2 = 0.0
        p_value = 1.0
    else:
        chi2 = ((abs(b - c) - 1) ** 2) / (b + c + 1e-9)
        z = sqrt(chi2)
        p_value = 2 * (1 - 0.5 * (1 + erf(z / sqrt(2))))

    return {
        "b": b,
        "c": c,
        "n": n,
        "chi2": float(chi2),
        "p_value": float(p_value)
    }


mcnemar_rows = []

for comparison_name, baseline_pred in [
    ("Best vs LSTM", y_pred_lstm),
    ("Best vs BiLSTM", y_pred_bilstm),
    ("Best vs Transformer", y_pred_transformer),
    ("Best vs DLSTM", y_pred_dlstm),
    ("Best vs Simple/Avg Ensemble", y_pred_simple),
    ("Best vs Stacking Ensemble", y_pred_stack),
]:
    out = mcnemar_test(y_test_seq, best_pred, baseline_pred)
    out["Comparison"] = comparison_name
    out["Best_Model"] = best_model_name
    mcnemar_rows.append(out)

mcnemar_df = pd.DataFrame(mcnemar_rows)[
    ["Comparison", "Best_Model", "b", "c", "n", "chi2", "p_value"]
]

display(mcnemar_df)

mcnemar_df.to_csv(
    os.path.join(OUT_DIR, "MCNEMAR_24H_BEST_MODEL_COMPARISONS.csv"),
    index=False
)

,Comparison,Best_Model,b,c,n,chi2,p_value
0,Best vs LSTM,BiLSTM,17,21,38,0.236842,0.626496
1,Best vs BiLSTM,BiLSTM,0,0,0,0.000000,1.000000
2,Best vs Transformer,BiLSTM,44,31,75,1.920000,0.165857
3,Best vs DLSTM,BiLSTM,31,21,52,1.557692,0.212003
4,Best vs Simple/Avg Ensemble,BiLSTM,12,11,23,0.000000,1.000000
5,Best vs Stacking Ensemble,BiLSTM,10,9,19,0.000000,1.000000


## Manuscript reminder

For the 24h section, report the clean table from:

`FINAL_CLEAN_24H_RESULTS_WITH_TRANSFORMER.csv`

Use the cadence diagnostics to describe the **actual input span**, and state clearly that:

- thresholds were tuned on validation only;
- weighted ensemble weights were selected using validation TSS only;
- the test set was reserved for final reporting.



---



In [18]:
X_train_scaled, X_val_scaled, X_test_scaled
y_train_seq, y_val_seq, y_test_seq
OUT_DIR

'/content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER'

In [21]:
import os, random, gc, itertools
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TUNE_OUT_DIR = os.path.join(OUT_DIR, "24h_all_models_hyperparameter_tuning")
os.makedirs(TUNE_OUT_DIR, exist_ok=True)

print("Saving tuning outputs to:", TUNE_OUT_DIR)
print("TensorFlow:", tf.__version__)


# ============================================================
# Metric helpers
# ============================================================

def tss_hss_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    tss = recall - fpr

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn) + 1e-9)
    hss = numerator / denominator

    return float(tss), float(hss)


def tune_threshold_by_tss(y_true, prob):
    thresholds = np.linspace(0, 1, 1001)

    best_thr = 0.5
    best_tss = -999

    for thr in thresholds:
        y_pred = (prob >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)

        if tss > best_tss:
            best_tss = tss
            best_thr = thr

    return float(best_thr), float(best_tss)


def evaluate_probs(model_name, y_val, val_prob, y_test, test_prob, extra=None):
    threshold, val_tss = tune_threshold_by_tss(y_val, val_prob)

    y_pred = (test_prob >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tss, hss = tss_hss_from_cm(cm)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "Model": model_name,
        "Threshold": threshold,
        "Val_TSS": val_tss,
        "Test_TSS": tss,
        "HSS": hss,
        "ROC_AUC": roc_auc_score(y_test, test_prob),
        "PR_AUC": average_precision_score(y_test, test_prob),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

    if extra is not None:
        result.update(extra)

    return result, y_pred


def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )

    return {int(c): float(w) for c, w in zip(classes, weights)}


def make_callbacks(patience=5):
    return [
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=0
        ),
        ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-5,
            verbose=0
        )
    ]


# ============================================================
# Decomposition helper for DLSTM
# ============================================================

def moving_average_decomposition(X, window=3):
    trend = np.zeros_like(X)
    kernel = np.ones(window, dtype=np.float32) / float(window)

    for feature_idx in range(X.shape[2]):
        for sample_idx in range(X.shape[0]):
            # Ensure the convoluted output matches the expected sequence length
            convolved_output = np.convolve(
                X[sample_idx, :, feature_idx],
                kernel,
                mode="same"
            )
            trend[sample_idx, :, feature_idx] = convolved_output[:X.shape[1]]

    residual = X - trend

    return trend, residual


# ============================================================
# Model builders
# ============================================================

def build_lstm_tuned(input_shape, lstm_units=64, dense_units=32, dropout=0.3, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.LSTM(
            lstm_units,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0
        ),

        layers.Dense(
            dense_units,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        ),

        layers.Dropout(dropout),

        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def build_bilstm_tuned(input_shape, lstm_units=64, dense_units=32, dropout=0.3, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Bidirectional(
            layers.LSTM(
                lstm_units,
                return_sequences=False,
                dropout=dropout,
                recurrent_dropout=0.0
            )
        ),

        layers.Dense(
            dense_units,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        ),

        layers.Dropout(dropout),

        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def build_dlstm_tuned(input_shape, lstm_units=64, dropout=0.3, lr=1e-3):
    second_units = max(16, lstm_units // 2)

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.LSTM(
            lstm_units,
            return_sequences=True,
            dropout=dropout,
            recurrent_dropout=0.0
        ),

        layers.Dropout(dropout),

        layers.LSTM(
            second_units,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0
        ),

        layers.Dropout(dropout),

        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


def transformer_encoder_block(x, num_heads=4, key_dim=32, ff_dim=128, dropout=0.2):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x


def build_transformer_tuned(
    input_shape,
    num_blocks=1,
    num_heads=2,
    key_dim=16,
    ff_dim=64,
    dense_units=32,
    dropout=0.2,
    lr=1e-3
):
    inp = layers.Input(shape=input_shape)

    x = layers.Dense(64, activation="relu")(inp)

    for _ in range(num_blocks):
        x = transformer_encoder_block(
            x,
            num_heads=num_heads,
            key_dim=key_dim,
            ff_dim=ff_dim,
            dropout=dropout
        )

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(
        dense_units,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(dropout)(x)

    out = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inp, out, name="Transformer_24h_Tuned")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    return model


# ============================================================
# Tuning grids
# Keep this moderate. Transformer tuning is expensive.
# ============================================================

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])
class_weight_tune = get_class_weights(y_train_seq)

print("Input shape:", input_shape)
print("Class weights:", class_weight_tune)

EPOCHS_TUNE = 25

LSTM_UNITS_GRID = [64, 128]
DENSE_UNITS_GRID = [32, 64]
DROPOUT_GRID = [0.2, 0.3]
LR_GRID = [1e-3, 5e-4]
BATCH_SIZE_GRID = [64, 128]

MA_WINDOW_GRID = [3, 5]

TRANSFORMER_BLOCKS_GRID = [1, 2]
TRANSFORMER_HEADS_GRID = [2, 4]
TRANSFORMER_KEY_DIM_GRID = [16, 32]
TRANSFORMER_FF_DIM_GRID = [64, 128]


# ============================================================
# Main tuning loop
# ============================================================

tuning_rows = []
prob_store = {}

run_id = 0


def run_and_record(model, model_name, family, batch_size, extra):
    global run_id

    history = model.fit(
        X_train_input,
        y_train_seq,
        validation_data=(X_val_input, y_val_seq),
        epochs=EPOCHS_TUNE,
        batch_size=batch_size,
        class_weight=class_weight_tune,
        callbacks=make_callbacks(patience=5),
        shuffle=False,
        verbose=0
    )

    val_prob = model.predict(X_val_input, verbose=0).ravel()
    test_prob = model.predict(X_test_input, verbose=0).ravel()

    result, y_pred = evaluate_probs(
        model_name,
        y_val_seq,
        val_prob,
        y_test_seq,
        test_prob,
        extra={
            "Family": family,
            "Batch_Size": batch_size,
            "Epochs_Used": len(history.history["loss"]),
            **extra
        }
    )

    tuning_rows.append(result)

    prob_store[model_name] = {
        "val_prob": val_prob,
        "test_prob": test_prob,
        "y_pred": y_pred,
        "result": result
    }

    print(
        f"Val_TSS={result['Val_TSS']:.4f} | "
        f"Test_TSS={result['Test_TSS']:.4f} | "
        f"PR_AUC={result['PR_AUC']:.4f} | "
        f"Precision={result['Precision']:.4f} | "
        f"F1={result['F1']:.4f}"
    )


# -----------------------------
# LSTM tuning
# -----------------------------
for lstm_units, dense_units, dropout, lr, batch_size in itertools.product(
    LSTM_UNITS_GRID,
    DENSE_UNITS_GRID,
    DROPOUT_GRID,
    LR_GRID,
    BATCH_SIZE_GRID
):
    run_id += 1

    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model_name = f"LSTM_24h_U{lstm_units}_Dense{dense_units}_D{dropout}_LR{lr}_BS{batch_size}"

    print("\n======================================")
    print(f"Run {run_id}: {model_name}")
    print("======================================")

    X_train_input = X_train_scaled
    X_val_input = X_val_scaled
    X_test_input = X_test_scaled

    model = build_lstm_tuned(
        input_shape=input_shape,
        lstm_units=lstm_units,
        dense_units=dense_units,
        dropout=dropout,
        lr=lr
    )

    run_and_record(
        model,
        model_name,
        "LSTM",
        batch_size,
        {
            "LSTM_Units": lstm_units,
            "Dense_Units": dense_units,
            "Dropout": dropout,
            "Learning_Rate": lr,
            "MA_Window": np.nan,
            "Transformer_Blocks": np.nan,
            "Transformer_Heads": np.nan,
            "Transformer_Key_Dim": np.nan,
            "Transformer_FF_Dim": np.nan
        }
    )

    del model
    gc.collect()


# -----------------------------
# BiLSTM tuning
# -----------------------------
for lstm_units, dense_units, dropout, lr, batch_size in itertools.product(
    LSTM_UNITS_GRID,
    DENSE_UNITS_GRID,
    DROPOUT_GRID,
    LR_GRID,
    BATCH_SIZE_GRID
):
    run_id += 1

    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model_name = f"BiLSTM_24h_U{lstm_units}_Dense{dense_units}_D{dropout}_LR{lr}_BS{batch_size}"

    print("\n======================================")
    print(f"Run {run_id}: {model_name}")
    print("======================================")

    X_train_input = X_train_scaled
    X_val_input = X_val_scaled
    X_test_input = X_test_scaled

    model = build_bilstm_tuned(
        input_shape=input_shape,
        lstm_units=lstm_units,
        dense_units=dense_units,
        dropout=dropout,
        lr=lr
    )

    run_and_record(
        model,
        model_name,
        "BiLSTM",
        batch_size,
        {
            "LSTM_Units": lstm_units,
            "Dense_Units": dense_units,
            "Dropout": dropout,
            "Learning_Rate": lr,
            "MA_Window": np.nan,
            "Transformer_Blocks": np.nan,
            "Transformer_Heads": np.nan,
            "Transformer_Key_Dim": np.nan,
            "Transformer_FF_Dim": np.nan
        }
    )

    del model
    gc.collect()


# -----------------------------
# DLSTM tuning
# -----------------------------
for ma_window, lstm_units, dropout, lr, batch_size in itertools.product(
    MA_WINDOW_GRID,
    LSTM_UNITS_GRID,
    DROPOUT_GRID,
    LR_GRID,
    BATCH_SIZE_GRID
):
    run_id += 1

    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model_name = f"DLSTM_24h_MA{ma_window}_U{lstm_units}_D{dropout}_LR{lr}_BS{batch_size}"

    print("\n======================================")
    print(f"Run {run_id}: {model_name}")
    print("======================================")

    trend_train, res_train = moving_average_decomposition(X_train_scaled, window=ma_window)
    trend_val, res_val = moving_average_decomposition(X_val_scaled, window=ma_window)
    trend_test, res_test = moving_average_decomposition(X_test_scaled, window=ma_window)

    X_train_input = np.concatenate([trend_train, res_train], axis=2)
    X_val_input = np.concatenate([trend_val, res_val], axis=2)
    X_test_input = np.concatenate([trend_test, res_test], axis=2)

    model = build_dlstm_tuned(
        input_shape=(X_train_input.shape[1], X_train_input.shape[2]),
        lstm_units=lstm_units,
        dropout=dropout,
        lr=lr
    )

    run_and_record(
        model,
        model_name,
        "DLSTM",
        batch_size,
        {
            "LSTM_Units": lstm_units,
            "Dense_Units": np.nan,
            "Dropout": dropout,
            "Learning_Rate": lr,
            "MA_Window": ma_window,
            "Transformer_Blocks": np.nan,
            "Transformer_Heads": np.nan,
            "Transformer_Key_Dim": np.nan,
            "Transformer_FF_Dim": np.nan
        }
    )

    del model
    gc.collect()


# -----------------------------
# Transformer tuning
# -----------------------------
for num_blocks, num_heads, key_dim, ff_dim, dropout, lr, batch_size in itertools.product(
    TRANSFORMER_BLOCKS_GRID,
    TRANSFORMER_HEADS_GRID,
    TRANSFORMER_KEY_DIM_GRID,
    TRANSFORMER_FF_DIM_GRID,
    DROPOUT_GRID,
    LR_GRID,
    BATCH_SIZE_GRID
):
    run_id += 1

    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model_name = (
        f"Transformer_24h_B{num_blocks}_H{num_heads}_K{key_dim}"
        f"_FF{ff_dim}_D{dropout}_LR{lr}_BS{batch_size}"
    )

    print("\n======================================")
    print(f"Run {run_id}: {model_name}")
    print("======================================")

    X_train_input = X_train_scaled
    X_val_input = X_val_scaled
    X_test_input = X_test_scaled

    model = build_transformer_tuned(
        input_shape=input_shape,
        num_blocks=num_blocks,
        num_heads=num_heads,
        key_dim=key_dim,
        ff_dim=ff_dim,
        dense_units=32,
        dropout=dropout,
        lr=lr
    )

    run_and_record(
        model,
        model_name,
        "Transformer",
        batch_size,
        {
            "LSTM_Units": np.nan,
            "Dense_Units": 32,
            "Dropout": dropout,
            "Learning_Rate": lr,
            "MA_Window": np.nan,
            "Transformer_Blocks": num_blocks,
            "Transformer_Heads": num_heads,
            "Transformer_Key_Dim": key_dim,
            "Transformer_FF_Dim": ff_dim
        }
    )

    del model
    gc.collect()


# ============================================================
# Collate and save results
# ============================================================

tuning_all_df = pd.DataFrame(tuning_rows)

tuning_by_val = tuning_all_df.sort_values(
    "Val_TSS",
    ascending=False
).reset_index(drop=True)

tuning_by_test = tuning_all_df.sort_values(
    "Test_TSS",
    ascending=False
).reset_index(drop=True)

print("\n======================================")
print("BEST CONFIGURATIONS BY VALIDATION TSS")
print("Official selection rule")
print("======================================")
display(tuning_by_val.head(15).round(4))

print("\n======================================")
print("AUDIT ONLY: TOP 15 BY TEST TSS")
print("Do NOT use this for model selection")
print("======================================")
display(tuning_by_test.head(15).round(4))

tuning_by_val.to_csv(
    os.path.join(TUNE_OUT_DIR, "24H_ALL_MODELS_TUNING_BY_VAL_TSS.csv"),
    index=False
)

tuning_by_test.to_csv(
    os.path.join(TUNE_OUT_DIR, "AUDIT_ONLY_24H_ALL_MODELS_TUNING_BY_TEST_TSS.csv"),
    index=False
)

# Save official best validation-selected probability arrays
best_val_model_name = tuning_by_val.iloc[0]["Model"]
best_val_item = prob_store[best_val_model_name]

np.save(os.path.join(TUNE_OUT_DIR, "best_val_24h_val_prob.npy"), best_val_item["val_prob"])
np.save(os.path.join(TUNE_OUT_DIR, "best_val_24h_test_prob.npy"), best_val_item["test_prob"])
np.save(os.path.join(TUNE_OUT_DIR, "best_val_24h_test_pred.npy"), best_val_item["y_pred"])

print("\nSaved tuning outputs to:", TUNE_OUT_DIR)
print("Best validation-selected model:", best_val_model_name)
print("\nOfficial best validation-selected result:")
display(pd.DataFrame([best_val_item["result"]]).round(4))


Saving tuning outputs to: /content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER/24h_all_models_hyperparameter_tuning
TensorFlow: 2.20.0
Input shape: (4, 21)
Class weights: {0: 0.5593856655290103, 1: 4.709770114942529}

Run 1: LSTM_24h_U64_Dense32_D0.2_LR0.001_BS64
Val_TSS=0.5719 | Test_TSS=0.4195 | PR_AUC=0.6693 | Precision=0.4684 | F1=0.5993

Run 2: LSTM_24h_U64_Dense32_D0.2_LR0.001_BS128
Val_TSS=0.5521 | Test_TSS=0.4551 | PR_AUC=0.6952 | Precision=0.4944 | F1=0.6175

Run 3: LSTM_24h_U64_Dense32_D0.2_LR0.0005_BS64
Val_TSS=0.5590 | Test_TSS=0.4690 | PR_AUC=0.6903 | Precision=0.5119 | F1=0.6255

Run 4: LSTM_24h_U64_Dense32_D0.2_LR0.0005_BS128
Val_TSS=0.5322 | Test_TSS=0.4515 | PR_AUC=0.6972 | Precision=0.5030 | F1=0.6159

Run 5: LSTM_24h_U64_Dense32_D0.3_LR0.001_BS64
Val_TSS=0.5804 | Test_TSS=0.4515 | PR_AUC=0.6745 | Precision=0.5030 | F1=0.6159

Run 6: LSTM_24h_U64_Dense32_D0.3_LR0.001_BS128
Val_TSS=0.5625 | Test_TSS=0.4825 | PR_AUC=0.6991 | Precision=0.5179 | F1=

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,Epochs_Used,LSTM_Units,Dense_Units,Dropout,Learning_Rate,MA_Window,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,0.599,0.6151,0.4806,0.4297,0.8086,0.6902,0.7664,0.5395,0.6332,...,25,64.0,32.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN
1,LSTM_24h_U64_Dense64_D0.3_LR0.001_BS64,0.578,0.6052,0.4364,0.3693,0.8160,0.6767,0.8037,0.4886,0.6078,...,18,64.0,64.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN
2,LSTM_24h_U64_Dense64_D0.3_LR0.0005_BS64,0.593,0.6047,0.4626,0.4032,0.8175,0.6819,0.7850,0.5153,0.6222,...,25,64.0,64.0,0.3,0.0005,NaN,NaN,NaN,NaN,NaN
3,BiLSTM_24h_U128_Dense64_D0.3_LR0.001_BS128,0.570,0.5982,0.4148,0.3495,0.8122,0.6900,0.7944,0.4775,0.5965,...,20,128.0,64.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN
4,DLSTM_24h_MA3_U128_D0.2_LR0.001_BS64,0.573,0.5962,0.4434,0.3795,0.8052,0.6730,0.7944,0.4971,0.6115,...,17,128.0,NaN,0.2,0.0010,3.0,NaN,NaN,NaN,NaN
5,DLSTM_24h_MA5_U128_D0.2_LR0.001_BS64,0.582,0.5962,0.4376,0.3679,0.8131,0.6812,0.8131,0.4860,0.6084,...,21,128.0,NaN,0.2,0.0010,5.0,NaN,NaN,NaN,NaN
6,BiLSTM_24h_U128_Dense64_D0.2_LR0.001_BS64,0.632,0.5957,0.4485,0.4029,0.8003,0.6810,0.7383,0.5267,0.6148,...,17,128.0,64.0,0.2,0.0010,NaN,NaN,NaN,NaN,NaN
7,LSTM_24h_U128_Dense64_D0.2_LR0.0005_BS64,0.666,0.5947,0.4380,0.3972,0.8096,0.6757,0.7196,0.5274,0.6087,...,21,128.0,64.0,0.2,0.0005,NaN,NaN,NaN,NaN,NaN
8,LSTM_24h_U64_Dense64_D0.2_LR0.0005_BS128,0.566,0.5947,0.4277,0.3525,0.8163,0.6820,0.8318,0.4734,0.6034,...,25,64.0,64.0,0.2,0.0005,NaN,NaN,NaN,NaN,NaN
9,BiLSTM_24h_U64_Dense64_D0.3_LR0.001_BS64,0.565,0.5928,0.4328,0.3738,0.8172,0.6970,0.7757,0.4970,0.6058,...,20,64.0,64.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN



AUDIT ONLY: TOP 15 BY TEST TSS
Do NOT use this for model selection


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,Epochs_Used,LSTM_Units,Dense_Units,Dropout,Learning_Rate,MA_Window,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,BiLSTM_24h_U64_Dense32_D0.2_LR0.0005_BS64,0.651,0.5848,0.4859,0.4324,0.8155,0.6980,0.7757,0.5390,0.6360,...,25,64.0,32.0,0.2,0.0005,NaN,NaN,NaN,NaN,NaN
1,LSTM_24h_U64_Dense32_D0.3_LR0.001_BS128,0.594,0.5625,0.4825,0.4157,0.8222,0.6991,0.8131,0.5179,0.6327,...,25,64.0,32.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN
2,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,0.599,0.6151,0.4806,0.4297,0.8086,0.6902,0.7664,0.5395,0.6332,...,25,64.0,32.0,0.3,0.0010,NaN,NaN,NaN,NaN,NaN
3,DLSTM_24h_MA5_U128_D0.3_LR0.001_BS64,0.571,0.5804,0.4779,0.4017,0.8188,0.6845,0.8411,0.5028,0.6294,...,22,128.0,NaN,0.3,0.0010,5.0,NaN,NaN,NaN,NaN
4,LSTM_24h_U128_Dense64_D0.2_LR0.001_BS128,0.657,0.5615,0.4766,0.4251,0.8158,0.6878,0.7664,0.5359,0.6308,...,15,128.0,64.0,0.2,0.0010,NaN,NaN,NaN,NaN,NaN
5,BiLSTM_24h_U128_Dense64_D0.3_LR0.0005_BS64,0.595,0.5858,0.4743,0.4068,0.8193,0.6983,0.8131,0.5118,0.6282,...,20,128.0,64.0,0.3,0.0005,NaN,NaN,NaN,NaN,NaN
6,DLSTM_24h_MA5_U128_D0.3_LR0.0005_BS64,0.609,0.5412,0.4738,0.3974,0.8112,0.6716,0.8411,0.5000,0.6272,...,23,128.0,NaN,0.3,0.0005,5.0,NaN,NaN,NaN,NaN
7,LSTM_24h_U64_Dense32_D0.3_LR0.0005_BS64,0.577,0.5561,0.4714,0.4007,0.8162,0.6900,0.8224,0.5057,0.6263,...,25,64.0,32.0,0.3,0.0005,NaN,NaN,NaN,NaN,NaN
8,LSTM_24h_U64_Dense32_D0.2_LR0.0005_BS64,0.582,0.5590,0.4690,0.4042,0.8170,0.6903,0.8037,0.5119,0.6255,...,25,64.0,32.0,0.2,0.0005,NaN,NaN,NaN,NaN,NaN
9,BiLSTM_24h_U64_Dense64_D0.3_LR0.0005_BS128,0.593,0.5650,0.4685,0.3947,0.8206,0.7074,0.8318,0.5000,0.6246,...,25,64.0,64.0,0.3,0.0005,NaN,NaN,NaN,NaN,NaN



Saved tuning outputs to: /content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER/24h_all_models_hyperparameter_tuning
Best validation-selected model: BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64

Official best validation-selected result:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,Epochs_Used,LSTM_Units,Dense_Units,Dropout,Learning_Rate,MA_Window,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,0.599,0.6151,0.4806,0.4297,0.8086,0.6902,0.7664,0.5395,0.6332,...,25,64,32,0.3,0.001,NaN,NaN,NaN,NaN,NaN


In [23]:
# ============================================================
# MCNEMAR TEST FOR 24H TUNED MODELS
# Compares official best validation-selected model
# against best validation-selected model from each family.
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy.stats import chi2

# -----------------------------
# McNemar helper
# -----------------------------
def mcnemar_test(y_true, pred_a, pred_b):
    """
    pred_a = official best model predictions
    pred_b = comparator model predictions
    """

    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # b: A correct, B wrong
    b = np.sum(correct_a & ~correct_b)

    # c: A wrong, B correct
    c = np.sum(~correct_a & correct_b)

    n = b + c

    if n == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        # continuity-corrected McNemar chi-square
        chi2_stat = ((abs(b - c) - 1) ** 2) / n
        p_value = chi2.sf(chi2_stat, df=1)

    return {
        "b": int(b),
        "c": int(c),
        "n": int(n),
        "chi2": float(chi2_stat),
        "p_value": float(p_value)
    }


# -----------------------------
# Pick official best model
# -----------------------------
official_best_row = tuning_by_val.iloc[0]
official_best_model = official_best_row["Model"]

official_best_pred = prob_store[official_best_model]["y_pred"]
y_true = np.array(y_test_seq).astype(int)

print("Official best tuned model:", official_best_model)


# -----------------------------
# Pick best validation-selected model per family
# -----------------------------
best_family_rows = (
    tuning_by_val
    .sort_values("Val_TSS", ascending=False)
    .groupby("Family", as_index=False)
    .first()
)

display(best_family_rows[[
    "Family", "Model", "Val_TSS", "Test_TSS",
    "ROC_AUC", "PR_AUC", "Recall", "Precision", "F1"
]].round(4))


# -----------------------------
# Run McNemar comparisons
# -----------------------------
mcnemar_rows = []

for _, row in best_family_rows.iterrows():
    comparator_model = row["Model"]
    comparator_family = row["Family"]

    if comparator_model == official_best_model:
        continue

    comparator_pred = prob_store[comparator_model]["y_pred"]

    result = mcnemar_test(
        y_true=y_true,
        pred_a=official_best_pred,
        pred_b=comparator_pred
    )

    result.update({
        "Comparison": f"Tuned Best BiLSTM vs Best Tuned {comparator_family}",
        "Official_Best_Model": official_best_model,
        "Comparator_Model": comparator_model,
        "Comparator_Family": comparator_family,
        "Interpretation": (
            "Significant difference"
            if result["p_value"] < 0.05
            else "No significant difference"
        )
    })

    mcnemar_rows.append(result)


mcnemar_tuned_df = pd.DataFrame(mcnemar_rows)

# Reorder columns
mcnemar_tuned_df = mcnemar_tuned_df[[
    "Comparison",
    "Official_Best_Model",
    "Comparator_Model",
    "Comparator_Family",
    "b", "c", "n", "chi2", "p_value", "Interpretation"
]]

display(mcnemar_tuned_df.round(6))

# Save
mcnemar_tuned_df.to_csv(
    os.path.join(TUNE_OUT_DIR, "MCNEMAR_24H_TUNED_MODEL_COMPARISONS.csv"),
    index=False
)

print("Saved tuned McNemar table to:", TUNE_OUT_DIR)

Official best tuned model: BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64


,Family,Model,Val_TSS,Test_TSS,ROC_AUC,PR_AUC,Recall,Precision,F1
0,BiLSTM,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,0.6151,0.4806,0.8086,0.6902,0.7664,0.5395,0.6332
1,DLSTM,DLSTM_24h_MA3_U128_D0.2_LR0.001_BS64,0.5962,0.4434,0.8052,0.6730,0.7944,0.4971,0.6115
2,LSTM,LSTM_24h_U64_Dense64_D0.3_LR0.001_BS64,0.6052,0.4364,0.8160,0.6767,0.8037,0.4886,0.6078
3,Transformer,Transformer_24h_B2_H2_K32_FF128_D0.2_LR0.001_BS64,0.5422,0.3325,0.7611,0.6256,0.7570,0.4378,0.5548


,Comparison,Official_Best_Model,Comparator_Model,Comparator_Family,b,c,n,chi2,p_value,Interpretation
0,Tuned Best BiLSTM vs Best Tuned DLSTM,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,DLSTM_24h_MA3_U128_D0.2_LR0.001_BS64,DLSTM,23,10,33,4.363636,0.036714,Significant difference
1,Tuned Best BiLSTM vs Best Tuned LSTM,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,LSTM_24h_U64_Dense64_D0.3_LR0.001_BS64,LSTM,26,10,36,6.250000,0.012419,Significant difference
2,Tuned Best BiLSTM vs Best Tuned Transformer,BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64,Transformer_24h_B2_H2_K32_FF128_D0.2_LR0.001_BS64,Transformer,51,16,67,17.253731,0.000033,Significant difference


Saved tuned McNemar table to: /content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER/24h_all_models_hyperparameter_tuning


In [24]:
# ============================================================
# BOOTSTRAP CI FOR OFFICIAL TUNED 24H BEST MODEL
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

def compute_tss_from_predictions(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)

    return float(recall - fpr)


# Official best validation-selected model
official_best_model = tuning_by_val.iloc[0]["Model"]
official_best_pred = prob_store[official_best_model]["y_pred"]

y_true = np.array(y_test_seq).astype(int)
y_pred = np.array(official_best_pred).astype(int)

BOOT_N = 2000
rng = np.random.default_rng(42)

boot_tss = []

for _ in range(BOOT_N):
    idx = rng.choice(len(y_true), size=len(y_true), replace=True)

    # Skip invalid bootstrap samples with only one class
    if len(np.unique(y_true[idx])) < 2:
        continue

    boot_tss.append(
        compute_tss_from_predictions(
            y_true[idx],
            y_pred[idx]
        )
    )

boot_tss = np.array(boot_tss)

bootstrap_24h_tuned_ci = {
    "model": official_best_model,
    "mean": float(np.mean(boot_tss)),
    "ci_low": float(np.percentile(boot_tss, 2.5)),
    "ci_high": float(np.percentile(boot_tss, 97.5)),
    "n_boot_valid": int(len(boot_tss))
}

print("Official tuned 24h model:", official_best_model)
print("Bootstrap TSS CI:", bootstrap_24h_tuned_ci)

# Save outputs
pd.DataFrame([bootstrap_24h_tuned_ci]).to_csv(
    os.path.join(TUNE_OUT_DIR, "BOOTSTRAP_24H_TUNED_BILSTM_TSS_CI.csv"),
    index=False
)

np.save(
    os.path.join(TUNE_OUT_DIR, "BOOTSTRAP_24H_TUNED_BILSTM_TSS_SCORES.npy"),
    boot_tss
)

print("Saved bootstrap CI to:", TUNE_OUT_DIR)

Official tuned 24h model: BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64
Bootstrap TSS CI: {'model': 'BiLSTM_24h_U64_Dense32_D0.3_LR0.001_BS64', 'mean': 0.48090434535426635, 'ci_low': 0.38185717573917305, 'ci_high': 0.5780781251383921, 'n_boot_valid': 2000}
Saved bootstrap CI to: /content/drive/MyDrive/AR_Stratified/Journal_Revision_24h_CLEAN_TRANSFORMER/24h_all_models_hyperparameter_tuning
